# 03. Вимір дат

Цей ноутбук відповідає за третій етап конвеєра. Він читає мінімальну та максимальну дату курсів із bronze, а потім створює безперервний календар.

У таблиці `dim_date` кожна дата повинна зустрічатися лише один раз. Календар міститиме не тільки дати, за які є курси валют, а всі дні відповідних років, включно з вихідними та святами.

Цей ноутбук не звертається до API НБУ. Він використовує таблицю `nbu_raw.raw_rates`, яку підготував перший ноутбук.

Результат записується в режимі `WRITE_TRUNCATE`. Під час кожного запуску календар будується заново на основі доступного діапазону дат.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Знайти мінімальну та максимальну дату в bronze.
3. Визначити перший і останній роки календаря.
4. Створити безперервний список дат.
5. Розрахувати календарні ознаки.
6. Додати технічний рядок `Unknown`.
7. Записати результат у `nbu_dwh.dim_date`.
8. Перевірити календар.

## 1. Налаштування та підключення

Вказуємо назву Google Cloud проєкту, імпортуємо потрібні бібліотеки та створюємо клієнт BigQuery.

Ноутбук підключається до BigQuery самостійно, оскільки надалі оркестратор запускатиме кожен етап окремо.

Для авторизації використовуємо JSON-ключ сервісного акаунта зі змінної середовища `GCP_SA_KEY`. Сам ключ у коді не зберігається.

In [1]:
PROJECT_ID = "nbu-bigquery-etl"
LOCATION = "EU"

DS_RAW = "nbu_raw"
DS_DWH = "nbu_dwh"

RAW_TABLE = f"{PROJECT_ID}.{DS_RAW}.raw_rates"
DIM_DATE_TABLE = f"{PROJECT_ID}.{DS_DWH}.dim_date"

In [2]:
import os
import sys
import json

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.max_columns", 40)

In [3]:
creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account

    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]

    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)

print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 4. Ноутбук 03: вимір дат

### Завдання 4.1. Визначення діапазону дат

Читаємо з bronze мінімальну та максимальну `business_date`.

Мінімальна дата показує початок доступного періоду, а максимальна дата показує його кінець. На основі років цих дат ми визначимо межі календаря.

На цьому етапі не завантажуємо всі рядки bronze в pandas. BigQuery сам знаходить мінімальну та максимальну дату й повертає тільки один підсумковий рядок.

In [4]:
sql = f"""
SELECT MIN(business_date) AS min_date,
       MAX(business_date) AS max_date
  FROM `{RAW_TABLE}`
"""

date_bounds = client.query(sql).to_dataframe()

date_bounds

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,min_date,max_date
0,2026-08-24,2026-08-24


### Завдання 4.2. Створення безперервного календаря

Визначаємо роки мінімальної та максимальної дати з bronze. Календар починаємо з 1 січня першого року і завершуємо 31 грудня останнього року.

Використовуємо `pd.date_range()` із частотою `D`, щоб отримати всі календарні дні поспіль. Завдяки цьому у вимірі будуть присутні вихідні, свята та дні, за які в bronze немає курсів.

In [5]:
year_min = date_bounds.loc[0, "min_date"].year
year_max = date_bounds.loc[0, "max_date"].year

dates = pd.date_range(start=f"{year_min}-01-01", end=f"{year_max}-12-31", freq="D")

print("кількість дат:", len(dates))
print("перша дата:", dates[0].date())
print("остання дата:", dates[-1].date())

кількість дат: 365
перша дата: 2026-01-01
остання дата: 2026-12-31


### Завдання 4.3. Додавання календарних ознак

На основі безперервного списку дат створюємо DataFrame виміру.

Для кожної дати розраховуємо числовий ключ, рік, квартал, місяць, поєднання року й місяця, назву дня тижня та ознаку вихідного дня.

Наприкінці переводимо `full_date` у звичайну дату без часу, щоб у BigQuery вона була записана як `DATE`, а не як `TIMESTAMP`.

In [6]:
dim = pd.DataFrame({
    "date_key": dates.strftime("%Y%m%d").astype(int),
    "full_date": dates,
    "year": dates.year,
    "quarter": dates.quarter,
    "month": dates.month,
    "year_month": dates.strftime("%Y-%m"),
    "day_name": dates.day_name(),
    "is_weekend": dates.weekday >= 5
})

dim["full_date"] = dim["full_date"].dt.date

dim.head(5)

,date_key,full_date,year,quarter,month,year_month,day_name,is_weekend
0,20260101,2026-01-01,2026,1,1,2026-01,Thursday,False
1,20260102,2026-01-02,2026,1,1,2026-01,Friday,False
2,20260103,2026-01-03,2026,1,1,2026-01,Saturday,True
3,20260104,2026-01-04,2026,1,1,2026-01,Sunday,True
4,20260105,2026-01-05,2026,1,1,2026-01,Monday,False


### Завдання 4.4. Додавання рядка Unknown

Додаємо технічний рядок для невідомої дати з ключем `-1`.

У `full_date` записуємо умовну дату `1900-01-01`. Для числових календарних ознак використовуємо `-1`, а для текстових полів використовуємо значення `Unknown`.

Такий рядок знадобиться, якщо дата з таблиці фактів не буде знайдена у вимірі дат. Він дозволяє не залишати ключ порожнім і не втрачати фактичний запис.

In [7]:
dim.loc[len(dim)] = {"date_key": -1, "full_date": pd.Timestamp("1900-01-01").date(), "year": -1,
                     "quarter": -1, "month": -1, "year_month": "Unknown", "day_name": "Unknown",
                     "is_weekend": False}

print(len(dim), "рядків після додавання Unknown")
dim.tail(5)

366 рядків після додавання Unknown


,date_key,full_date,year,quarter,month,year_month,day_name,is_weekend
361,20261228,2026-12-28,2026,4,12,2026-12,Monday,False
362,20261229,2026-12-29,2026,4,12,2026-12,Tuesday,False
363,20261230,2026-12-30,2026,4,12,2026-12,Wednesday,False
364,20261231,2026-12-31,2026,4,12,2026-12,Thursday,False
365,-1,1900-01-01,-1,-1,-1,Unknown,Unknown,False


### Завдання 4.5. Запис виміру дат у BigQuery

Записуємо підготовлений календар у таблицю `nbu_dwh.dim_date`.

Схему таблиці задаємо явно, щоб назви й типи колонок були визначені заздалегідь. Усі поля позначаємо як `REQUIRED`, оскільки вимір дат не повинен містити порожніх значень.

Для запису використовуємо режим `WRITE_TRUNCATE`. Під час першого запуску BigQuery створить таблицю, а під час наступних запусків повністю замінить її вміст новим календарем.

Повторний запуск із тим самим діапазоном років повинен створити такий самий результат без дублікатів.

In [8]:
dim_date_schema = [
    bigquery.SchemaField("date_key",   "INTEGER",  mode="REQUIRED"),
    bigquery.SchemaField("full_date",  "DATE",     mode="REQUIRED"),
    bigquery.SchemaField("year",       "INTEGER",  mode="REQUIRED"),
    bigquery.SchemaField("quarter",    "INTEGER",  mode="REQUIRED"),
    bigquery.SchemaField("month",      "INTEGER",  mode="REQUIRED"),
    bigquery.SchemaField("year_month", "STRING",   mode="REQUIRED"),
    bigquery.SchemaField("day_name",   "STRING",   mode="REQUIRED"),
    bigquery.SchemaField("is_weekend", "BOOLEAN",  mode="REQUIRED"),
]

cfg = bigquery.LoadJobConfig(schema=dim_date_schema, write_disposition="WRITE_TRUNCATE")

load_job = client.load_table_from_dataframe(dim, DIM_DATE_TABLE, job_config=cfg)

load_job.result()

table = client.get_table(DIM_DATE_TABLE)

print("таблиця:", DIM_DATE_TABLE)
print("записано рядків:", table.num_rows)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


таблиця: nbu-bigquery-etl.nbu_dwh.dim_date
записано рядків: 366


### Завдання 4.6. Перевірка виміру дат

Читаємо записану таблицю `nbu_dwh.dim_date` із BigQuery та перевіряємо дві умови.

Спочатку перевіряємо безперервність календаря. Для цього виключаємо технічний рядок `Unknown`, сортуємо звичайні дати та обчислюємо різницю між кожними двома сусідніми датами. Кожна різниця повинна дорівнювати одному дню.

Також перевіряємо, що в таблиці присутній технічний рядок із `date_key = -1`.

In [9]:
sql = f"""
SELECT date_key,
       full_date,
       year, quarter,
       month, year_month,
       day_name,
       is_weekend
  FROM `{DIM_DATE_TABLE}`
ORDER BY date_key
"""

check_dim = client.query(sql).to_dataframe()

print(len(check_dim), f"рядків прочитано з {DIM_DATE_TABLE}")
check_dim.head(5)

366 рядків прочитано з nbu-bigquery-etl.nbu_dwh.dim_date


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_key,full_date,year,quarter,month,year_month,day_name,is_weekend
0,-1,1900-01-01,-1,-1,-1,Unknown,Unknown,False
1,20260101,2026-01-01,2026,1,1,2026-01,Thursday,False
2,20260102,2026-01-02,2026,1,1,2026-01,Friday,False
3,20260103,2026-01-03,2026,1,1,2026-01,Saturday,True
4,20260104,2026-01-04,2026,1,1,2026-01,Sunday,True


In [10]:
calendar_dates = (check_dim.loc[check_dim["date_key"] != -1,"full_date"].sort_values())

# Для першої дати різницю обчислити неможливо, бо перед нею немає попередньої дати.
# Тому pandas повертає порожнє значення, яке прибираємо через dropna().
date_differences = calendar_dates.diff().dropna()

print("у календарі немає пропущених днів:", date_differences.eq(pd.Timedelta(days=1)).all())

print("рядок із date_key = -1 існує:", (check_dim["date_key"] == -1).any())

у календарі немає пропущених днів: True
рядок із date_key = -1 існує: True


In [11]:
print("03_dim_date завершено успішно")

03_dim_date завершено успішно
